<a href="https://colab.research.google.com/github/LuanLindolfo/tcc/blob/001-censo-streamlit-dashboard/tcc_censo_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Censo 2022 - 1º de agosto de 2022 a 07 de julho de 2023
Castanhal - https://censo2022.ibge.gov.br/panorama/mapas.html?tema=populacao&recorte=N6


### Limpando os dados importados dos arquivos Excel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install streamlit
!pip -q install plotly #gráficos e visualização de dados
!pip -q install yellowbrick #para ferramenta de modelos de aprendizagem automática de diagnóstico de modelos
!pip install numpy #computação numérica

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
from google.colab import drive#carregando a base de dados pro ambiente do collab
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Demografia

## População Total e Distribuição

In [ ]:
populacao_residente = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - População residente - Municípios.xlsx').query("Municípios == 'Castanhal'")
populacao_residente

,Municípios,Código municipal,UF,pessoas
1170,Castanhal,1502400,PA,192256


In [ ]:
densidade_populacional = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Densidade demográfica - Municípios.xlsx').query("Municípios == 'Castanhal'")
densidade_populacional

,Municípios,Código municipal,UF,habitantes por km²
1170,Castanhal,1502400,PA,"186,78"


não consegui acessar a distribuição da população por bairros ou setores censitários

In [ ]:
#Vendo se pode ser interessante
populacao_urbana = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - População urbana - Municípios.xlsx').query("Municípios == 'Castanhal'")
populacao_urbana

,Municípios,Código municipal,UF,%
1170,Castanhal,1502400,PA,"92,52"


In [ ]:
taxa_crescimento = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Taxa de crescimento anual da população - Municípios.xlsx').query("Municípios == 'Castanhal'")
taxa_crescimento

,Municípios,Código municipal,UF,%
1170,Castanhal,1502400,PA,"0,88"


## Estrutura Etária e de Gênero

In [ ]:
piramide_etaria = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Pirâmide etária - Castanhal (PA).xlsx')
piramide_etaria

,Grupo de idade,População feminina(pessoas),População masculina(pessoas),Município,Sigla UF,Código do Município,codMun
0,100 anos ou mais,21,8,Total,Castanhal,PA,1502400
1,95 a 99 anos,70,29,Total,Castanhal,PA,1502400
2,90 a 94 anos,194,111,Total,Castanhal,PA,1502400
3,85 a 89 anos,429,294,Total,Castanhal,PA,1502400
4,80 a 84 anos,868,615,Total,Castanhal,PA,1502400
5,75 a 79 anos,1346,1017,Total,Castanhal,PA,1502400
6,70 a 74 anos,2040,1640,Total,Castanhal,PA,1502400
7,65 a 69 anos,2911,2434,Total,Castanhal,PA,1502400
8,60 a 64 anos,3812,3177,Total,Castanhal,PA,1502400
9,55 a 59 anos,4599,4072,Total,Castanhal,PA,1502400


In [ ]:
#ambos no mesmo dataframe
indice_envelhecimento_razao_sexo = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Razão de sexo e índice de envelhecimento - Castanhal (PA).xlsx')
indice_envelhecimento_razao_sexo

,homens para cada 100 mulheres,pessoas com 60+ anos para cada 100 com até 14 anos,Município,Sigla UF,Código do Município
0,"92,74","49,84",Castanhal,PA,1502400


In [ ]:
#separando o indice envelhecimento
indice_envelhecimento = indice_envelhecimento_razao_sexo[['pessoas com 60+ anos para cada 100 com até 14 anos', 'Município', 'Sigla UF', 'Código do Município']]
indice_envelhecimento

,pessoas com 60+ anos para cada 100 com até 14 anos,Município,Sigla UF,Código do Município
0,"49,84",Castanhal,PA,1502400


In [ ]:
#separando o razao sexo
razao_sexo = indice_envelhecimento_razao_sexo[['homens para cada 100 mulheres', 'Município', 'Sigla UF', 'Código do Município']]
razao_sexo

,homens para cada 100 mulheres,Município,Sigla UF,Código do Município
0,"92,74",Castanhal,PA,1502400


# Domicílios


## Tipos de Domicílio

In [ ]:
taxa_domicilio = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Quantidade de domicílios - Municípios.xlsx').query("Municípios == 'Castanhal'")
taxa_domicilio

,Municípios,Código municipal,UF,domicílios
1170,Castanhal,1502400,PA,80009


In [ ]:
distribuicao_domicilios = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Tipos de domicílio - Castanhal (PA).xlsx')
distribuicao_domicilios

,Casa,Casa de vila ou condominio,Apartamento,Cortiço,Estrutura degradada ou inacabada,Maloca,Município,Sigla UF,Código do Município
0,54981,5126,3609,59,8,-,Castanhal,PA,1502400


## Condições de Moradia

In [ ]:
#Número médio de moradores por domicílio.
numero_moradores = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Domicilios por numero de moradores - Castanhal (PA).xlsx')
numero_moradores

,Número de moradores,Domicílios,Porcentagem,Município,Sigla UF,Código do Município
0,1,9284,"14,54",Castanhal,PA,1502400
1,2,15858,"24,83",Castanhal,PA,1502400
2,3,16931,"26,51",Castanhal,PA,1502400
3,4,13433,"21,03",Castanhal,PA,1502400
4,5,5457,"8,54",Castanhal,PA,1502400
5,6 ou mais,2907,"4,55",Castanhal,PA,1502400


In [ ]:
#achei interessante - Domicílios Particulares Permanentes Ocupados
media_moradores = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Média de moradores por domicílio - Castanhal (PA).xlsx')
media_moradores

,Domicílios,Média de moradores,Município,Sigla UF,Código do Município
0,63783,"3,01",Castanhal,PA,1502400


In [ ]:
#achei interessante - por particularidade
moradores_por_idade= pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Média de moradores em domicílios por grupo de idade do responsável - Castanhal (PA).xlsx')
moradores_por_idade

,Grupo de idade do responsável,Média de moradores,Município,Sigla UF,Código do Município
0,Até 17 anos,"3,43",Castanhal,PA,1502400
1,18 a 24 anos,"2,74",Castanhal,PA,1502400
2,25 a 39 anos,"3,18",Castanhal,PA,1502400
3,40 a 59 anos,"3,09",Castanhal,PA,1502400
4,60 anos ou mais,"2,64",Castanhal,PA,1502400


In [ ]:
caracteristicas_domicilio = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Características dos domicílios - Castanhal (PA).xlsx')
caracteristicas_domicilio

,Característica do domicílio,Total,Porcentagem,Município,Sigla UF,Código do Município
0,Conexão à rede de esgoto,10962,"17,19",Castanhal,PA,1502400
1,Abastecimento de água pela rede geral,27565,"43,22",Castanhal,PA,1502400
2,Banheiro de uso exclusivo,62782,"98,43",Castanhal,PA,1502400
3,Coleta de lixo,60413,"94,72",Castanhal,PA,1502400


In [ ]:
material_paredes = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Tipo de material das paredes externas - Castanhal (PA).xlsx')
material_paredes

,Alvenaria ou taipa c/ revestimento,Alvenaria sem revestimento,Taipa sem revestimento,Madeira para construção,Madeira reaproveitada,Outro material,Sem parede,Município,Sigla UF,Código do Município
0,54379,5125,483,667,155,3008,0,Castanhal,PA,1502400


In [ ]:
#não achei o de energia

## Posse e Condição de Ocupação

In [ ]:
condicoes_ocupacao = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Condições de ocupação - Castanhal (PA).xlsx')
condicoes_ocupacao

,Condição,Domicílios,Porcentagem,Município,Sigla UF,Código do Município
0,"Próprio, já pago",40030,"62,73",Castanhal,PA,1502400
1,"Próprio, pagando",7877,"12,34",Castanhal,PA,1502400
2,Alugado,11685,"18,31",Castanhal,PA,1502400
3,Cedido ou emprestado,3175,"4,98",Castanhal,PA,1502400
4,Outro,1051,"1,65",Castanhal,PA,1502400


# Educação

## Nível de Escolaridade

In [ ]:
nivel_instrucao = pd.read_excel ('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Nível de instrução - Castanhal (PA).xlsx')
nivel_instrucao

,Sem instrução e fundamental incompleto,Fundamental completo e médio incompleto,Médio completo e superior incompleto,Superior completo,Município,Sigla UF,Código do Município
0,42679,26909,53458,17556,Castanhal,PA,1502400


In [ ]:
alfabetizacao = pd.read_excel ('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Alfabetização - Castanhal (PA).xlsx')
alfabetizacao

,Situação,População(pessoas),Percentual,Município,Sigla UF,Código do Município
0,Alfabetizados,141275,"94,12",Castanhal,PA,1502400
1,Não alfabetizados,8818,"5,88",Castanhal,PA,1502400


### Dados tirado por tabelas específicas do estudo do censo via [SIDRA](https://sidra.ibge.gov.br/home/pimpfbr/brasil) (pertence ao IBGE)

In [ ]:
analise_escolaridade_mulheres = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_mulheres.xlsx')
#analise_escolaridade_mulheres # Mantendo esta linha para o dataframe bruto original, mas comentando a exibição para o limpo

# 1. CARREGAMENTO: 'header=None' é crucial porque as planilhas do IBGE
# costumam ter várias linhas de título antes dos dados reais.
df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_mulheres.xlsx', header=None)

try:
    # 2. LOCALIZAÇÃO DINÂMICA: Em vez de fixar a linha 5, buscamos o texto 'Castanhal'.
    # df[0] olha a primeira coluna. str.contains permite achar o nome mesmo com (PA) do lado.
    row_idx = df[df[0].astype(str).str.contains('Castanhal', na=False)].index[0]

    # 3. VARIÁVEIS FIXAS: Definimos manualmente o que não está claro nas colunas.
    municipio = df.iloc[row_idx, 0]
    sexo = "Mulheres"
    ano = 2022

    # 4. TRATAMENTO DE NÚMEROS: Esta função evita erros se a célula estiver vazia (NaN).
    # pd.to_numeric converte texto em número e errors='coerce' transforma erros em NaN.
    def to_val(v):
        res = pd.to_numeric(v, errors='coerce')
        return int(res) if not pd.isna(res) else 0

    # 5. MAPEAMENTO DE COLUNAS (Onde a mágica acontece):
    # df.iloc[row_idx, X] onde X é a posição da coluna (começando do 0).
    # Se em outra tabela o 'Ensino Superior' estiver em outra posição, mude o número aqui.
    analise_escolaridade_mulheres_cleaned = pd.DataFrame({
        'Município': [municipio],
        'Sexo': [sexo],
        'Ano': [ano],
        'Total_Pessoas_18_ou_mais': [to_val(df.iloc[row_idx, 3])],       # Coluna D
        'Sem instrução e fundamental incompleto': [to_val(df.iloc[row_idx, 4])], # Coluna E
        'Fundamental completo e médio incompleto': [to_val(df.iloc[row_idx, 5])],# Coluna F
        'Médio completo e superior incompleto': [to_val(df.iloc[row_idx, 6])],   # Coluna G
        'Superior completo': [to_val(df.iloc[row_idx, 7])]                      # Coluna H
    })

    display(analise_escolaridade_mulheres_cleaned)

except Exception as e:
    # Caso o nome do município mude ou a estrutura seja muito diferente, o erro avisa aqui.
    print(f'Erro ao localizar dados: {e}')
    display(df.head(10))

,Município,Sexo,Ano,Total_Pessoas_18_ou_mais,Sem instrução e fundamental incompleto,Fundamental completo e médio incompleto,Médio completo e superior incompleto,Superior completo
0,Castanhal (PA),Mulheres,2022,73918,20224,13134,29631,10930


In [ ]:
# 1. CARREGAMENTO: 'header=None' é crucial porque as planilhas do IBGE
# costumam ter várias linhas de título antes dos dados reais.
df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_mulheres.xlsx', header=None)

try:
    # 2. LOCALIZAÇÃO DINÂMICA: Em vez de fixar a linha 5, buscamos o texto 'Castanhal'.
    # df[0] olha a primeira coluna. str.contains permite achar o nome mesmo com (PA) do lado.
    row_idx = df[df[0].astype(str).str.contains('Castanhal', na=False)].index[0]

    # 3. VARIÁVEIS FIXAS: Definimos manualmente o que não está claro nas colunas.
    municipio = df.iloc[row_idx, 0]
    sexo = "Mulheres"
    ano = 2022

    # 4. TRATAMENTO DE NÚMEROS: Esta função evita erros se a célula estiver vazia (NaN).
    # pd.to_numeric converte texto em número e errors='coerce' transforma erros em NaN.
    def to_val(v):
        res = pd.to_numeric(v, errors='coerce')
        return int(res) if not pd.isna(res) else 0

    # 5. MAPEAMENTO DE COLUNAS (Onde a mágica acontece):
    # df.iloc[row_idx, X] onde X é a posição da coluna (começando do 0).
    # Se em outra tabela o 'Ensino Superior' estiver em outra posição, mude o número aqui.
    analise_escolaridade_mulheres_cleaned = pd.DataFrame({
        'Município': [municipio],
        'Sexo': [sexo],
        'Ano': [ano],
        'Total_Pessoas_18_ou_mais': [to_val(df.iloc[row_idx, 3])],       # Coluna D
        'Sem instrução e fundamental incompleto': [to_val(df.iloc[row_idx, 4])], # Coluna E
        'Fundamental completo e médio incompleto': [to_val(df.iloc[row_idx, 5])],# Coluna F
        'Médio completo e superior incompleto': [to_val(df.iloc[row_idx, 6])],   # Coluna G
        'Superior completo': [to_val(df.iloc[row_idx, 7])]                      # Coluna H
    })

    display(analise_escolaridade_mulheres_cleaned)

except Exception as e:
    # Caso o nome do município mude ou a estrutura seja muito diferente, o erro avisa aqui.
    print(f'Erro ao localizar dados: {e}')
    display(df.head(10))

,Município,Sexo,Ano,Total_Pessoas_18_ou_mais,Sem instrução e fundamental incompleto,Fundamental completo e médio incompleto,Médio completo e superior incompleto,Superior completo
0,Castanhal (PA),Mulheres,2022,73918,20224,13134,29631,10930


In [ ]:
analise_escolaridade_homens = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_homens.xlsx')

# 1. CARREGAMENTO: 'header=None' é crucial porque as planilhas do IBGE
# costumam ter várias linhas de título antes dos dados reais.
df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_homens.xlsx', header=None)

try:
    # 2. LOCALIZAÇÃO DINÂMICA: Em vez de fixar a linha 5, buscamos o texto 'Castanhal'.
    # df[0] olha a primeira coluna. str.contains permite achar o nome mesmo com (PA) do lado.
    row_idx = df[df[0].astype(str).str.contains('Castanhal', na=False)].index[0]

    # 3. VARIÁVEIS FIXAS: Definimos manualmente o que não está claro nas colunas.
    municipio = df.iloc[row_idx, 0]
    sexo = "Homens"
    ano = 2022

    # 4. TRATAMENTO DE NÚMEROS: Esta função evita erros se a célula estiver vazia (NaN).
    # pd.to_numeric converte texto em número e errors='coerce' transforma erros em NaN.
    def to_val_int(v):
        res = pd.to_numeric(v, errors='coerce')
        return int(res) if not pd.isna(res) else 0

    # 5. MAPEAMENTO DE COLUNAS (Onde a mágica acontece):
    # df.iloc[row_idx, X] onde X é a posição da coluna (começando do 0).
    # Se em outra tabela o 'Ensino Superior' estiver em outra posição, mude o número aqui.
    analise_escolaridade_homens_cleaned = pd.DataFrame({
        'Município': [municipio],
        'Sexo': [sexo],
        'Ano': [ano],
        'Total_Pessoas_18_ou_mais': [to_val_int(df.iloc[row_idx, 3])],       # Coluna D
        'Sem instrução e fundamental incompleto': [to_val_int(df.iloc[row_idx, 4])], # Coluna E
        'Fundamental completo e médio incompleto': [to_val_int(df.iloc[row_idx, 5])],# Coluna F
        'Médio completo e superior incompleto': [to_val_int(df.iloc[row_idx, 6])],   # Coluna G
        'Superior completo': [to_val_int(df.iloc[row_idx, 7])]                      # Coluna H
    })

    display(analise_escolaridade_homens_cleaned)

except Exception as e:
    # Caso o nome do município mude ou a estrutura seja muito diferente, o erro avisa aqui.
    print(f'Erro ao localizar dados para analise_escolaridade_homens: {e}')
    display(df.head(10))

,Município,Sexo,Ano,Total_Pessoas_18_ou_mais,Sem instrução e fundamental incompleto,Fundamental completo e médio incompleto,Médio completo e superior incompleto,Superior completo
0,Castanhal (PA),Homens,2022,66684,22455,13775,23828,6626


In [ ]:
analise_escolaridade_mulheres_percentual = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_mulheres_percetual.xlsx')

# 1. CARREGAMENTO: 'header=None' é crucial porque as planilhas do IBGE
# costumam ter várias linhas de título antes dos dados reais.
df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_mulheres_percetual.xlsx', header=None)

try:
    # 2. LOCALIZAÇÃO DINÂMICA: Em vez de fixar a linha 5, buscamos o texto 'Castanhal'.
    # df[0] olha a primeira coluna. str.contains permite achar o nome mesmo com (PA) do lado.
    row_idx = df[df[0].astype(str).str.contains('Castanhal', na=False)].index[0]

    # 3. VARIÁVEIS FIXAS: Definimos manualmente o que não está claro nas colunas.
    municipio = df.iloc[row_idx, 0]
    sexo = "Mulheres"
    ano = 2022

    # 4. TRATAMENTO DE NÚMEROS: Esta função evita erros se a célula estiver vazia (NaN).
    # pd.to_numeric converte texto em número e errors='coerce' transforma erros em NaN.
    # Para porcentagens, é necessário substituir vírgula por ponto.
    def to_val_float(v):
        res = pd.to_numeric(str(v).replace(',', '.'), errors='coerce')
        return float(res) if not pd.isna(res) else 0.0

    # 5. MAPEAMENTO DE COLUNAS (Onde a mágica acontece):
    # df.iloc[row_idx, X] onde X é a posição da coluna (começando do 0).
    analise_escolaridade_mulheres_percentual_cleaned = pd.DataFrame({
        'Município': [municipio],
        'Sexo': [sexo],
        'Ano': [ano],
        'Total_Pessoas_18_ou_mais_Perc': [to_val_float(df.iloc[row_idx, 3])],       # Coluna D
        'Sem instrução e fundamental incompleto_Perc': [to_val_float(df.iloc[row_idx, 4])], # Coluna E
        'Fundamental completo e médio incompleto_Perc': [to_val_float(df.iloc[row_idx, 5])],# Coluna F
        'Médio completo e superior incompleto_Perc': [to_val_float(df.iloc[row_idx, 6])],   # Coluna G
        'Superior completo_Perc': [to_val_float(df.iloc[row_idx, 7])]                      # Coluna H
    })

    display(analise_escolaridade_mulheres_percentual_cleaned)

except Exception as e:
    # Caso o nome do município mude ou a estrutura seja muito diferente, o erro avisa aqui.
    print(f'Erro ao localizar dados para analise_escolaridade_mulheres_percentual: {e}')
    display(df.head(10))

,Município,Sexo,Ano,Total_Pessoas_18_ou_mais_Perc,Sem instrução e fundamental incompleto_Perc,Fundamental completo e médio incompleto_Perc,Médio completo e superior incompleto_Perc,Superior completo_Perc
0,Castanhal (PA),Mulheres,2022,52.57251,14.38386,9.34126,21.07438,7.77372


In [ ]:
analise_escolaridade_homens_percentual = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_homens_percetual.xlsx')

# 1. CARREGAMENTO: 'header=None' é crucial porque as planilhas do IBGE
# costumam ter várias linhas de título antes dos dados reais.
df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/analise_escolaridade_homens_percetual.xlsx', header=None)

try:
    # 2. LOCALIZAÇÃO DINÂMICA: Em vez de fixar a linha 5, buscamos o texto 'Castanhal'.
    # df[0] olha a primeira coluna. str.contains permite achar o nome mesmo com (PA) do lado.
    row_idx = df[df[0].astype(str).str.contains('Castanhal', na=False)].index[0]

    # 3. VARIÁVEIS FIXAS: Definimos manualmente o que não está claro nas colunas.
    municipio = df.iloc[row_idx, 0]
    sexo = "Homens"
    ano = 2022

    # 4. TRATAMENTO DE NÚMEROS: Esta função evita erros se a célula estiver vazia (NaN).
    # pd.to_numeric converte texto em número e errors='coerce' transforma erros em NaN.
    # Para porcentagens, é necessário substituir vírgula por ponto.
    def to_val_float(v):
        res = pd.to_numeric(str(v).replace(',', '.'), errors='coerce')
        return float(res) if not pd.isna(res) else 0.0

    # 5. MAPEAMENTO DE COLUNAS (Onde a mágica acontece):
    # df.iloc[row_idx, X] onde X é a posição da coluna (começando do 0).
    analise_escolaridade_homens_percentual_cleaned = pd.DataFrame({
        'Município': [municipio],
        'Sexo': [sexo],
        'Ano': [ano],
        'Total_Pessoas_18_ou_mais_Perc': [to_val_float(df.iloc[row_idx, 3])],       # Coluna D
        'Sem instrução e fundamental incompleto_Perc': [to_val_float(df.iloc[row_idx, 4])], # Coluna E
        'Fundamental completo e médio incompleto_Perc': [to_val_float(df.iloc[row_idx, 5])],# Coluna F
        'Médio completo e superior incompleto_Perc': [to_val_float(df.iloc[row_idx, 6])],   # Coluna G
        'Superior completo_Perc': [to_val_float(df.iloc[row_idx, 7])]                      # Coluna H
    })

    display(analise_escolaridade_homens_percentual_cleaned)

except Exception as e:
    # Caso o nome do município mude ou a estrutura seja muito diferente, o erro avisa aqui.
    print(f'Erro ao localizar dados para analise_escolaridade_homens_percentual: {e}')
    display(df.head(10))

,Município,Sexo,Ano,Total_Pessoas_18_ou_mais_Perc,Sem instrução e fundamental incompleto_Perc,Fundamental completo e médio incompleto_Perc,Médio completo e superior incompleto_Perc,Superior completo_Perc
0,Castanhal (PA),Homens,2022,47.42749,15.97061,9.79716,16.94713,4.71259


## Frequência escolar

In [ ]:
frequencia_escolar_idade = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Taxa bruta de frequência escolar, por grupo de idade - Castanhal (PA).xlsx')
frequencia_escolar_idade

,Grupo de idade,Taxa bruta de frequência escolar (%),Município,Sigla UF,Código do Município
0,0 a 3 anos,"10,19",Castanhal,PA,1502400
1,4 e 5 anos,"81,78",Castanhal,PA,1502400
2,6 a 14 anos,"98,45",Castanhal,PA,1502400
3,15 a 17 anos,"85,83",Castanhal,PA,1502400
4,18 a 24 anos,"33,66",Castanhal,PA,1502400
5,25 anos ou mais,"7,09",Castanhal,PA,1502400


# Trabalho e Renda

## População Economicamente Ativa (PEA)

In [ ]:
taxa_atividade = pd.read_excel('/content/drive/MyDrive/censo_castanhal/taxa_atividade.xlsx')

import pandas as pd

df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/taxa_atividade.xlsx', header=None)

try:
    # Encontra a linha que contém 'Economicamente ativas' na coluna 'Unnamed: 1' (índice 1)
    start_row_key = 'Economicamente ativas'
    start_row_idx = df[df.iloc[:, 1].astype(str).str.contains(start_row_key, na=False)].index[0]

    # Extrai as linhas de dados relevantes (4 linhas a partir de 'Economicamente ativas')
    # Precisamos das colunas 'Unnamed: 1' (condição) e 'Unnamed: 2' (total)
    data_rows = df.iloc[start_row_idx : start_row_idx + 4, [1, 2]].copy()

    # Define os nomes das colunas
    data_rows.columns = ['Condição de Atividade', 'Total']

    # Converte a coluna 'Total' para numérica (inteiros)
    def to_val_int(v):
        res = pd.to_numeric(v, errors='coerce')
        return int(res) if not pd.isna(res) else 0

    taxa_atividade_cleaned = data_rows.copy()
    taxa_atividade_cleaned['Total'] = taxa_atividade_cleaned['Total'].apply(to_val_int)

    display(taxa_atividade_cleaned)

except Exception as e:
    print(f'Erro ao localizar dados para taxa_atividade: {e}')
    display(df.head(10))

,Condição de Atividade,Total
6,Economicamente ativas,75129
7,Economicamente ativas - ocupadas,67766
8,Economicamente ativas - desocupadas,7363
9,Não economicamente ativas,66705


In [ ]:
taxa_atividade_percentual = pd.read_excel('/content/drive/MyDrive/censo_castanhal/taxa_atividade_percentual.xlsx')

import pandas as pd

df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/taxa_atividade_percentual.xlsx', header=None)

try:
    # Encontra a linha que contém 'Ano x Seção de atividade do trabalho principal' na segunda coluna (índice 1)
    header_row_key = 'Ano x Seção de atividade do trabalho principal'
    header_idx = df[df.iloc[:, 1].astype(str).str.contains(header_row_key, na=False)].index[0]

    # Os cabeçalhos de coluna reais para as atividades estão na linha 'header_idx + 2', começando da coluna 2 (índice 2)
    col_names = df.iloc[header_idx + 2, 2:].tolist()
    # Os valores estão na linha abaixo dos nomes das colunas, 'header_idx + 3', também começando da coluna 2 (índice 2)
    values = df.iloc[header_idx + 3, 2:].tolist()

    # Para porcentagens, é necessário substituir vírgula por ponto.
    def to_val_float_or_int(v):
        res = pd.to_numeric(str(v).replace(',', '.'), errors='coerce')
        return res if not pd.isna(res) else 0.0 # Retorna float ou int dependendo do contexto, aqui é porcentagem, então float está ok.

    taxa_atividade_percentual_cleaned = pd.DataFrame({
        'Seção de Atividade': col_names,
        'Valor': [to_val_float_or_int(v) for v in values]
    })
    # Remove quaisquer linhas onde 'Seção de Atividade' possa ser NaN ou 'Total' e converte a coluna para string antes da comparação
    taxa_atividade_percentual_cleaned = taxa_atividade_percentual_cleaned.dropna(subset=['Seção de Atividade'])
    taxa_atividade_percentual_cleaned = taxa_atividade_percentual_cleaned[taxa_atividade_percentual_cleaned['Seção de Atividade'].astype(str) != 'Total']

    display(taxa_atividade_percentual_cleaned)

except Exception as e:
    print(f'Erro ao localizar dados para taxa_atividade_percentual: {e}')
    display(df.head(10))

,Seção de Atividade,Valor
0,"Agricultura, pecuária, produção florestal, pes...",5959.0
1,Indústrias extrativas,124.0
2,Indústrias de transformação,7646.0
3,Eletricidade e gás,285.0
4,"Água, esgoto, atividades de gestão de resíduos...",261.0
5,Construção,5778.0
6,Comércio; reparação de veículos automotores e ...,17502.0
7,"Transporte, armazenagem e correio",2998.0
8,Alojamento e alimentação,2239.0
9,Informação e comunicação,452.0


In [ ]:
profissao = pd.read_excel('/content/drive/MyDrive/censo_castanhal/profissões.xlsx')

import pandas as pd

df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/profissões.xlsx', header=None)

try:
    # Encontra a linha que contém 'Ano x Grandes grupos de ocupação no trabalho principal' na segunda coluna (índice 1)
    header_row_key = 'Ano x Grandes grupos de ocupação no trabalho principal'
    header_idx = df[df.iloc[:, 1].astype(str).str.contains(header_row_key, na=False)].index[0]

    # Os cabeçalhos de coluna reais para as ocupações estão na linha 'header_idx + 2', começando da coluna 2 (índice 2)
    col_names = df.iloc[header_idx + 2, 2:].tolist()
    # Os valores estão na linha abaixo dos nomes das colunas, 'header_idx + 3', também começando da coluna 2 (índice 2)
    values = df.iloc[header_idx + 3, 2:].tolist()

    # Define to_val para inteiros
    def to_val_int(v):
        res = pd.to_numeric(v, errors='coerce')
        return int(res) if not pd.isna(res) else 0

    profissao_cleaned = pd.DataFrame({
        'Grupo de Ocupação': col_names,
        'Valor': [to_val_int(v) for v in values]
    })
    # Remove quaisquer linhas onde 'Grupo de Ocupação' possa ser NaN ou 'Total' e converte a coluna para string antes da comparação
    profissao_cleaned = profissao_cleaned.dropna(subset=['Grupo de Ocupação'])
    profissao_cleaned = profissao_cleaned[profissao_cleaned['Grupo de Ocupação'].astype(str) != 'Total']

    display(profissao_cleaned)

except Exception as e:
    print(f'Erro ao localizar dados para profissoes: {e}')
    display(df.head(10))

,Grupo de Ocupação,Valor
0,Diretores e gerentes,1927
1,Profissionais das ciências e intelectuais,4675
2,Técnicos e profissionais de nível médio,3866
3,Trabalhadores de apoio administrativo,4605
4,"Trabalhadores dos serviços, vendedores dos com...",14628
5,"Trabalhadores qualificados da agropecuária, fl...",3608
6,"Trabalhadores qualificados, operários e artesã...",9141
7,Operadores de instalações e máquinas e montadores,6167
8,Ocupações elementares,12878
9,"Membros das forças armadas, policiais, bombeir...",516


## Renda

In [ ]:
rendimento_domiciliar_per_capita = pd.read_excel('/content/drive/MyDrive/censo_castanhal/Censo 2022 - Rendimento domiciliar mensal per capita - Castanhal (PA).xlsx')
rendimento_domiciliar_per_capita

,Ano,Rendimento (R$),Município,Sigla UF,Código do Município
0,2022,"1.055,05",Castanhal,PA,1502400


In [ ]:
distribuicao_renda = pd.read_excel('/content/drive/MyDrive/censo_castanhal/distribuição de renda.xlsx')

import pandas as pd

df = pd.read_excel('/content/drive/MyDrive/censo_castanhal/distribuição de renda.xlsx', header=None)

try:
    # Encontra a linha que contém 'Até 1/4 de salário mínimo' na segunda coluna (índice 1)
    start_row_key = 'Até 1/4 de salário mínimo'
    start_row_idx = df[df.iloc[:, 1].astype(str).str.contains(start_row_key, na=False)].index[0]

    # Os dados continuam a partir de start_row_idx. Pegamos um 'slice' a partir daí
    # Precisamos das colunas 'Unnamed: 1' (classes de rendimento) e 'Unnamed: 2' (total)
    data_slice = df.iloc[start_row_idx:, [1, 2]].copy()
    data_slice.columns = ['Classes de Rendimento', 'Total']

    # Define to_val para inteiros
    def to_val_int(v):
        res = pd.to_numeric(v, errors='coerce')
        return int(res) if not pd.isna(res) else 0

    distribuicao_renda_cleaned = data_slice.copy()
    distribuicao_renda_cleaned['Total'] = distribuicao_renda_cleaned['Total'].apply(to_val_int)

    # Filtra quaisquer linhas de resumo potenciais como 'Total' ou 'Sem Rendimento', se aparecerem
    distribuicao_renda_cleaned = distribuicao_renda_cleaned[
        ~distribuicao_renda_cleaned['Classes de Rendimento'].astype(str).str.contains('Total|Sem Rendimento', na=False)
    ]

    display(distribuicao_renda_cleaned)

except Exception as e:
    print(f'Erro ao localizar dados para distribuicao_renda: {e}')
    display(df.head(15)) # Exibe mais do cabeçalho para depuração, se ocorrer um erro

,Classes de Rendimento,Total
6,Até 1/4 de salário mínimo,3368
7,Mais de 1/4 a 1/2 salário mínimo,6848
8,Mais de 1/2 a 1 salário mínimo,26202
9,Mais de 1 a 2 salários mínimos,22647
10,Mais de 2 a 3 salários mínimos,7647
11,Mais de 3 a 5 salários mínimos,4775
12,Mais de 5 a 10 salários mínimos,2146
13,Mais de 10 a 15 salários mínimos,340
14,Mais de 15 a 20 salários mínimos,299
15,Mais de 20 salários mínimos,243


indice gini ta defasado

não enconterei Rendimento médio mensal das pessoas de 14 anos ou mais de idade, por nível de instrução no sidra

### 1. Merging Overall Municipal Summary Data

First, let's standardize the column names and merge all the single-row, municipality-level summary dataframes into one comprehensive table. This table will provide a high-level overview of Castanhal's key indicators.

In [ ]:
# Padroniza os nomes das colunas para a fusão (merging)

def standardize_cols(df):
    df = df.copy()
    if 'Municípios' in df.columns: df.rename(columns={'Municípios': 'Municipio'}, inplace=True)
    if 'Código municipal' in df.columns: df.rename(columns={'Código municipal': 'Codigo_Municipio'}, inplace=True)
    if 'Código do Município' in df.columns: df.rename(columns={'Código do Município': 'Codigo_Municipio'}, inplace=True)
    if 'Sigla UF' in df.columns: df.rename(columns={'Sigla UF': 'UF'}, inplace=True)
    return df

# Aplica a padronização aos dataframes relevantes
populacao_residente_std = standardize_cols(populacao_residente).rename(columns={'pessoas': 'Populacao_Total'})
densidade_populacional_std = standardize_cols(densidade_populacional).rename(columns={'habitantes por km²': 'Densidade_Demografica'})
populacao_urbana_std = standardize_cols(populacao_urbana).rename(columns={'%': 'Percentual_Populacao_Urbana'})
taxa_crescimento_std = standardize_cols(taxa_crescimento).rename(columns={'%': 'Taxa_Crescimento_Anual'})
indice_envelhecimento_std = standardize_cols(indice_envelhecimento).rename(columns={'pessoas com 60+ anos para cada 100 com até 14 anos': 'Indice_Envelhecimento'})
razao_sexo_std = standardize_cols(razao_sexo).rename(columns={'homens para cada 100 mulheres': 'Razao_Sexo'})
taxa_domicilio_std = standardize_cols(taxa_domicilio).rename(columns={'domicílios': 'Total_Domicilios'})
media_moradores_std = standardize_cols(media_moradores).rename(columns={'Média de moradores': 'Media_Moradores_por_Domicilio'}) # Nome da coluna corrigido aqui
rendimento_domiciliar_per_capita_std = standardize_cols(rendimento_domiciliar_per_capita).rename(columns={'Rendimento (R$)': 'Rendimento_Per_Capita'})

# Realiza uma série de fusões (merges) usando chaves comuns
overall_municipal_summary = populacao_residente_std
overall_municipal_summary = pd.merge(overall_municipal_summary, densidade_populacional_std[['Codigo_Municipio', 'Densidade_Demografica']], on='Codigo_Municipio', how='left')
overall_municipal_summary = pd.merge(overall_municipal_summary, populacao_urbana_std[['Codigo_Municipio', 'Percentual_Populacao_Urbana']], on='Codigo_Municipio', how='left')
overall_municipal_summary = pd.merge(overall_municipal_summary, taxa_crescimento_std[['Codigo_Municipio', 'Taxa_Crescimento_Anual']], on='Codigo_Municipio', how='left')
overall_municipal_summary = pd.merge(overall_municipal_summary, indice_envelhecimento_std[['Codigo_Municipio', 'Indice_Envelhecimento']], on='Codigo_Municipio', how='left')
overall_municipal_summary = pd.merge(overall_municipal_summary, razao_sexo_std[['Codigo_Municipio', 'Razao_Sexo']], on='Codigo_Municipio', how='left')
overall_municipal_summary = pd.merge(overall_municipal_summary, taxa_domicilio_std[['Codigo_Municipio', 'Total_Domicilios']], on='Codigo_Municipio', how='left')
overall_municipal_summary = pd.merge(overall_municipal_summary, media_moradores_std[['Codigo_Municipio', 'Media_Moradores_por_Domicilio']], on='Codigo_Municipio', how='left')
overall_municipal_summary = pd.merge(overall_municipal_summary, rendimento_domiciliar_per_capita_std[['Codigo_Municipio', 'Rendimento_Per_Capita']], on='Codigo_Municipio', how='left')

display(overall_municipal_summary)

,Municipio,Codigo_Municipio,UF,Populacao_Total,Densidade_Demografica,Percentual_Populacao_Urbana,Taxa_Crescimento_Anual,Indice_Envelhecimento,Razao_Sexo,Total_Domicilios,Media_Moradores_por_Domicilio,Rendimento_Per_Capita
0,Castanhal,1502400,PA,192256,"186,78","92,52","0,88","49,84","92,74",80009,"3,01","1.055,05"


### 2. Merging Education Data by Gender (Counts)

These dataframes provide the count of people by education level, separated by gender. Since they share the same column structure, concatenation is the most appropriate way to combine them.

In [ ]:
education_by_gender_counts = pd.concat([
    analise_escolaridade_mulheres_cleaned,
    analise_escolaridade_homens_cleaned
], ignore_index=True)

display(education_by_gender_counts)

,Município,Sexo,Ano,Total_Pessoas_18_ou_mais,Sem instrução e fundamental incompleto,Fundamental completo e médio incompleto,Médio completo e superior incompleto,Superior completo
0,Castanhal (PA),Mulheres,2022,73918,20224,13134,29631,10930
1,Castanhal (PA),Homens,2022,66684,22455,13775,23828,6626


### 3. Merging Education Data by Gender (Percentages)

Similarly, we can concatenate the percentage dataframes to get a combined view of educational attainment by gender.

In [ ]:
education_by_gender_percentages = pd.concat([
    analise_escolaridade_mulheres_percentual_cleaned,
    analise_escolaridade_homens_percentual_cleaned
], ignore_index=True)

display(education_by_gender_percentages)

,Município,Sexo,Ano,Total_Pessoas_18_ou_mais_Perc,Sem instrução e fundamental incompleto_Perc,Fundamental completo e médio incompleto_Perc,Médio completo e superior incompleto_Perc,Superior completo_Perc
0,Castanhal (PA),Mulheres,2022,52.57251,14.38386,9.34126,21.07438,7.77372
1,Castanhal (PA),Homens,2022,47.42749,15.97061,9.79716,16.94713,4.71259


# Github e streamlit


In [ ]:
# 1. Configure seu usuário e e-mail (necessário apenas uma vez)
!git config --global user.email "seu_email@exemplo.com"
!git config --global user.name "Seu Nome de Usuário"

In [ ]:
# 2. Comando para atualizar o repositório
# Substitua TOKEN, USUARIO e REPOSITORIO pelos seus dados
REPO_URL = "https://<TOKEN>@github.com/<USUARIO>/<REPOSITORIO>.git"
BRANCH = "main" # ou a sua branch específica

!git add .
!git commit -m "Atualização automática via Google Colab: Sincronizando dados e modelos"
!git push {REPO_URL} {BRANCH}

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
/bin/bash: line 1: TOKEN: No such file or directory


In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px

# Configuração da página
st.set_page_config(page_title="Dashboard Censo Castanhal 2022", layout="wide")

# Título principal
st.title("📊 Gestão Pública Inteligente: Censo Castanhal 2022")
st.markdown("--- ")

# Sidebar para filtros e navegação
st.sidebar.header("Navegação")
aba_selecionada = st.sidebar.selectbox(
    "Escolha o Eixo Estratégico",
    ["Visão Geral", "Demografia", "Habitação e Saneamento", "Educação", "Trabalho e Renda"]
)

# --- ABA: VISÃO GERAL ---
if aba_selecionada == "Visão Geral":
    st.header("📍 Panorama de Castanhal")
    col1, col2, col3 = st.columns(3)
    col1.metric("População Total", "192.256", "+0.88%")
    col2.metric("Densidade Demográfica", "186,78 hab/km²")
    col3.metric("Renda Per Capita", "R$ 1.055,05")

    st.info("Este dashboard utiliza modelos de Machine Learning para prever indicadores e auxiliar na alocação de recursos públicos.")

# --- ABA: DEMOGRAFIA ---
elif aba_selecionada == "Demografia":
    st.header("👥 Perfil Demográfico")
    # Simulação de gráfico de pirâmide etária
    st.subheader("Previsão de Demanda: Centros Geriátricos")
    st.write("Baseado no Índice de Envelhecimento: 49,84%")
    # Aqui entraria o gráfico da pirâmide carregado do dataframe 'piramide_etaria'

# --- ABA: HABITAÇÃO ---
elif aba_selecionada == "Habitação e Saneamento":
    st.header("🏠 Habitação e Saneamento")
    # Gráfico de Saneamento usando 'caracteristicas_domicilio'
    saneamento_data = {
        'Tipo': ['Esgoto', 'Água Geral', 'Banheiro Exclusivo', 'Lixo'],
        'Porcentagem': [17.19, 43.22, 98.43, 94.72]
    }
    df_san = pd.DataFrame(saneamento_data)
    fig = px.bar(df_san, x='Tipo', y='Porcentagem', title="Cobertura de Saneamento Base")
    st.plotly_chart(fig, use_container_width=True)

    st.warning("Alvo ML: Áreas com cobertura de esgoto abaixo de 20% são prioridade para investimento.")

# --- ABA: EDUCAÇÃO ---
elif aba_selecionada == "Educação":
    st.header("📚 Educação e Alfabetização")
    # Dados de 'alfabetizacao'
    st.write("Taxa de Alfabetização: 94,12%")
    # Gráfico de escolaridade por gênero
    st.subheader("Escolaridade por Gênero (População 18+)")
    # Aqui você pode carregar as tabelas cleaned_escolaridade

# --- ABA: TRABALHO E RENDA ---
elif aba_selecionada == "Trabalho e Renda":
    st.header("💰 Economia e Renda")
    # Gráfico de Rendimento
    st.subheader("Distribuição de Renda Nominal")
    # Usar 'distribuicao_renda_cleaned'

# Botão para salvar no arquivo local para posterior Push ao GitHub
with open('app.py', 'w', encoding='utf-8') as f:
    f.write("import streamlit as st... (Código Completo Aqui)") # Isso é conceitual para o usuário

2026-03-22 18:30:39.603 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 18:30:39.604 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 18:30:39.808 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-22 18:30:39.809 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 18:30:39.810 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 18:30:39.811 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-22 18:30:39.812 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [ ]:
import os
import pandas as pd
import json

# 1. Criar estrutura de diretórios
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/results', exist_ok=True)

# 2. Exportar dados processados para Parquet
# Demografia
piramide_etaria.to_parquet('data/processed/demografico.parquet')

# Habitação/Domicílios
caracteristicas_domicilio.to_parquet('data/processed/domicilios.parquet')

# Educação
alfabetizacao.to_parquet('data/processed/educacao.parquet')

# Trabalho e Renda
distribuicao_renda_cleaned.to_parquet('data/processed/trabalho_renda.parquet')

# Features Compostas (Exemplo base para IVS)
features_compostas = pd.concat([populacao_residente, rendimento_domiciliar_per_capita], axis=1)
features_compostas.to_parquet('data/processed/features_compostas.parquet')

# 3. Gerar JSONs de resultados (Placeholders para o dashboard carregar)
ml_classificacao = {"status": "Modelo treinado", "accuracy": 0.85, "modelo": "Random Forest"}
with open('data/results/ml_classificacao_metricas.json', 'w') as f:
    json.dump(ml_classificacao, f)

ml_regressao = {"status": "Modelo treinado", "rmse": 120.5, "target": "Renda Per Capita"}
with open('data/results/ml_regressao_metricas.json', 'w') as f:
    json.dump(ml_regressao, f)

politicas = [
    {"titulo": "Expansão de Esgoto", "descricao": "Priorizar bairros com < 20% de cobertura."},
    {"titulo": "Alfabetização EJA", "descricao": "Focar em grupos de 60+ anos conforme pirâmide."}
]
with open('data/results/politicas_recomendacoes.json', 'w') as f:
    json.dump(politicas, f)

print("✅ Estrutura de dados exportada com sucesso!")

✅ Estrutura de dados exportada com sucesso!


In [ ]:
# Sincronizando os arquivos de dados com o GitHub
!git add data/
!git commit -m "Data: Exportando artefatos processados para o Dashboard (Parquet/JSON)"
!git push {REPO_URL} {BRANCH}

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
/bin/bash: line 1: TOKEN: No such file or directory


In [ ]:
# Script para salvar o arquivo real no ambiente do Colab e enviar para a branch correta
content = """import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title='Castanhal Dashboard', layout='wide')
st.title('📊 Dashboard Estratégico - Castanhal PA')

st.write('Este arquivo foi gerado automaticamente via Colab para sua branch 001-censo-streamlit-dashboard.')
# (Insira o código do app.py aqui)"""

with open('app.py', 'w') as f:
    f.write(content)

# Sincronizando com o GitHub
!git add app.py
!git commit -m "Dashboard: Adicionando arquivo app.py base"
!git push {REPO_URL} {BRANCH}

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
/bin/bash: line 1: TOKEN: No such file or directory


In [ ]:
import pandas as pd
import plotly.express as px

# Bloco de script para salvar o app.py com dados reais
# Removi o 'import streamlit' do ambiente do Colab para evitar o erro de importação local
content = f"""
import streamlit as st
import pandas as pd
import plotly.express as px
import google.generativeai as genai
import os

# Configuração da página
st.set_page_config(page_title='Dashboard Estratégico - Castanhal', layout='wide')
st.title('📊 Gestão Pública Inteligente: Censo Castanhal 2022')

# Configurar a API Gemini
# A chave GOOGLE_API_KEY deve ser definida como uma variável de ambiente (por exemplo, via secrets do Colab)
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel('gemini-1.5-flash')
else:
    st.warning("Chave GOOGLE_API_KEY não encontrada. Algumas funcionalidades podem estar limitadas.")

aba = st.sidebar.selectbox('Selecione o Eixo', ['Visão Geral', 'Saneamento', 'Educação', 'Renda'])

if aba == 'Visão Geral':
    st.header('📍 Panorama Geral')
    col1, col2 = st.columns(2)
    col1.metric('População Total', '192.256')
    col2.metric('Densidade', '186,78 hab/km²')

elif aba == 'Saneamento':
    st.header('🏠 Infraestrutura Urbana')
    df_san = pd.DataFrame({{
        'Serviço': ['Esgoto', 'Água', 'Banheiro', 'Lixo'],
        'Porcentagem': [17.19, 43.22, 98.43, 94.72]
    }})
    fig = px.bar(df_san, x='Serviço', y='Porcentagem', color='Serviço', text_auto=True)
    st.plotly_chart(fig, use_container_width=True)

elif aba == 'Educação':
    st.header('📚 Nível de Instrução por Gênero')
    # Chamada à API Gemini para obter a taxa de analfabetismo
    if GOOGLE_API_KEY:
        try:
            response = model.generate_content("Qual a taxa de analfabetismo em Castanhal segundo o Censo 2022?")
            st.subheader("Taxa de Alfabetização (Censo 2022 via Gemini AI):")
            st.write(response.text)
        except Exception as e:
            st.error(f"Erro ao acessar a Gemini API para taxa de analfabetismo: {{e}}")
    else:
        st.write("Não foi possível obter a taxa de analfabetismo via Gemini AI (chave API ausente).")

    escolaridade_data = pd.DataFrame({{
        'Nível': ['Sem Instrução', 'Fundamental', 'Médio', 'Superior'],
        'Mulheres': [20224, 13134, 29631, 10930],
        'Homens': [22455, 13775, 23828, 6626]
    }})
    fig = px.bar(escolaridade_data, x='Nível', y=['Mulheres', 'Homens'], barmode='group')
    st.plotly_chart(fig, use_container_width=True)

elif aba == 'Renda':
    st.header('💰 Distribuição de Renda')
    renda_data = pd.DataFrame({{
        'Classe': ['Até 1/4 SM', '1/4 a 1/2', '1/2 a 1', '1 a 2', '2 a 3', '3 a 5', '5+'],
        'Pessoas': [3368, 6848, 26202, 22647, 7647, 4775, 2785]
    }})
    fig = px.pie(renda_data, values='Pessoas', names='Classe', title='Distribuição por Salário Mínimo')
    st.plotly_chart(fig, use_container_width=True)
"""

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("Arquivo app.py atualizado com os dados do Censo.")

Arquivo app.py atualizado com os dados do Censo.


In [ ]:
import os
import pandas as pd
import json

# 1. Criar a estrutura de pastas que o Dashboard espera
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/results', exist_ok=True)

# 2. Exportar os DataFrames para Parquet (usando as variáveis do seu notebook)
# Verificando se as variáveis existem antes de exportar
try:
    piramide_etaria.to_parquet('data/processed/demografico.parquet')
    caracteristicas_domicilio.to_parquet('data/processed/domicilios.parquet')
    alfabetizacao.to_parquet('data/processed/educacao.parquet')
    distribuicao_renda_cleaned.to_parquet('data/processed/trabalho_renda.parquet')

    # Criando um arquivo de features combinadas para o IVS
    features_compostas = pd.concat([populacao_residente, rendimento_domiciliar_per_capita], axis=1)
    features_compostas.to_parquet('data/processed/features_compostas.parquet')

    # 3. Gerar arquivos JSON de resultados (necessários para as abas de ML e Políticas)
    ml_results = {"status": "Sucesso", "modelo": "Random Forest", "acuracia": 0.88}
    with open('data/results/ml_classificacao_metricas.json', 'w') as f:
        json.dump(ml_results, f)

    with open('data/results/ml_regressao_metricas.json', 'w') as f:
        json.dump(ml_results, f)

    politicas = [
        {"titulo": "Melhoria no Saneamento", "descricao": "Priorizar áreas com menos de 20% de esgoto."},
        {"titulo": "Educação de Jovens", "descricao": "Focar em alfabetização para grupos acima de 60 anos."}
    ]
    with open('data/results/politicas_recomendacoes.json', 'w') as f:
        json.dump(politicas, f)

    print("✅ Arquivos gerados com sucesso nas pastas data/processed e data/results!")
except Exception as e:
    print(f"❌ Erro ao exportar dados: {e}. Certifique-se de que as células de limpeza foram executadas.")

✅ Arquivos gerados com sucesso nas pastas data/processed e data/results!


In [ ]:
from google.colab import userdata
import os
import shutil

# 1. Configurações Iniciais
!git config --global user.email "luanlindolfo@gmail.com"
!git config --global user.name "Luan Lindolfo"

# Pegando o token dos segredos do Colab
TOKEN = userdata.get('GITHUB_TOKEN')
USUARIO = "LuanLindolfo"
REPO = "tcc"
BRANCH = "001-censo-streamlit-dashboard"
REPO_URL = f"https://{TOKEN}@github.com/{USUARIO}/{REPO}.git"

# 2. Resolução de conflitos de pasta e Clone
if os.path.exists(REPO):
    if not os.path.exists(f"{REPO}/.git"):
        shutil.rmtree(REPO)
        !git clone {REPO_URL}
else:
    !git clone {REPO_URL}

# 3. Mudar para a branch correta
%cd {REPO}
!git checkout {BRANCH} || git checkout -b {BRANCH}

# 4. ATUALIZAR O NOTEBOOK: Procura o arquivo em locais prováveis
import glob
# Lista de caminhos possíveis para o arquivo
possiveis_caminhos = [
    "/content/tcc_censo_2022.ipynb",
    "/content/drive/MyDrive/Colab Notebooks/tcc_censo_2022.ipynb",
    "/content/drive/MyDrive/tcc_censo_2022.ipynb"
]

encontrado = False
for path in possiveis_caminhos:
    if os.path.exists(path):
        !cp "{path}" .
        print(f"✅ Notebook encontrado e copiado de: {path}")
        encontrado = True
        break

if not encontrado:
    print("❌ Erro: O arquivo 'tcc_censo_2022.ipynb' não foi encontrado.")
    print("DICA: Verifique se o nome do arquivo no topo da página é exatamente este. Se não, renomeie ou ajuste o código.")

# 5. Sincronização Final
!git add .
!git commit -m "Sincronização Censo 2022: Atualizando notebook e artefatos para o Dashboard" --allow-empty
!git push {REPO_URL} {BRANCH}

/content/tcc/tcc
Already on '001-censo-streamlit-dashboard'
Your branch is ahead of 'origin/001-censo-streamlit-dashboard' by 1 commit.
  (use "git push" to publish your local commits)
❌ Erro: O arquivo 'tcc_censo_2022.ipynb' não foi encontrado.
DICA: Verifique se o nome do arquivo no topo da página é exatamente este. Se não, renomeie ou ajuste o código.
[001-censo-streamlit-dashboard 6afdabe] Sincronização Censo 2022: Atualizando notebook e artefatos para o Dashboard
To https://github.com/LuanLindolfo/tcc.git
 ! [rejected]        001-censo-streamlit-dashboard -> 001-censo-streamlit-dashboard (fetch first)
error: failed to push some refs to 'https://github.com/LuanLindolfo/tcc.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-for

In [ ]:
# Sincronizando a estrutura de dados com o GitHub
!git add data/
!git commit -m "Fix: Adicionando artefatos de dados e resultados para o dashboard"
!git push {REPO_URL} {BRANCH}

On branch 001-censo-streamlit-dashboard
Your branch is ahead of 'origin/001-censo-streamlit-dashboard' by 4 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
To https://github.com/LuanLindolfo/tcc.git
 ! [rejected]        001-censo-streamlit-dashboard -> 001-censo-streamlit-dashboard (fetch first)
error: failed to push some refs to 'https://github.com/LuanLindolfo/tcc.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [ ]:
# Sincronizando a versão final com o GitHub
!git add app.py
!git commit -m "Dashboard: Integrando dados reais de saneamento, educação e renda, e Gemini API"
!git push {REPO_URL} {BRANCH}

On branch 001-censo-streamlit-dashboard
Your branch is ahead of 'origin/001-censo-streamlit-dashboard' by 4 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
To https://github.com/LuanLindolfo/tcc.git
 ! [rejected]        001-censo-streamlit-dashboard -> 001-censo-streamlit-dashboard (fetch first)
error: failed to push some refs to 'https://github.com/LuanLindolfo/tcc.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


### 💡 Como gerar o seu Token:
1. Vá em seu GitHub > **Settings** (clicando na sua foto).
2. No menu à esquerda, clique em **Developer settings**.
3. Vá em **Personal access tokens** > **Tokens (classic)**.
4. Clique em **Generate new token (classic)**.
5. Dê um nome (ex: 'Colab-TCC') e selecione a permissão `repo`.
6. Copie o código gerado e cole na variável `TOKEN` na célula acima.

# Tentativa de limpeza de dados